In [1]:
!pip install -q -U langgraph langchain langchain-groq


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [2]:
# Install required packages
%pip install -U langgraph langchain-groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Import required modules
import os
from getpass import getpass
from typing import TypedDict
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

In [4]:
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")


In [5]:
# Import ChatGroq
from langchain_groq import ChatGroq

# Create Groq LLM using GPT-OSS
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

# Send prompt
response = llm.invoke("Explain LangGraph in one sentence")

# Print response
print(response.content)

LangGraph is a framework that lets developers design, orchestrate, and execute complex language‑model workflows as directed graphs, enabling modular, stateful, and context‑aware AI applications.


### FIRST STATE GRAPH

In [6]:
# Generate an AI topic using the LLM
topic = llm.invoke("Generate an AI topic").content

# Generate a 100-word article on the generated topic
article = llm.invoke(f"Write 100 words about {topic}").content

# Print the generated article
print(article)

Federated learning (FL) enables hospitals to collaboratively train deep‑learning models on sensitive medical images—MRIs, CT scans—without exchanging raw data. By aggregating locally computed gradients, FL preserves patient confidentiality while harnessing diverse datasets. Communication‑efficient protocols, such as sparsified updates and secure aggregation, reduce bandwidth demands. Differential privacy mechanisms inject calibrated noise, guaranteeing formal privacy bounds. Moreover, FL must handle heterogeneous data distributions across institutions, employing techniques like personalized models or domain adaptation to maintain robustness. The resulting models achieve higher diagnostic accuracy, meeting regulatory standards like HIPAA and GDPR, and fostering trust in AI‑driven medical imaging for patient care and research.


In [7]:
# Define the structure of the graph state
class State(TypedDict):
    topic: str

In [8]:
# Define a node that generates an AI topic
def topic_generator(state):
    # Ask the LLM to generate an AI topic
    topic = llm.invoke("Generate an AI topic").content
    # Return the generated topic to update the state
    return {"topic": topic}

In [9]:
# Create a StateGraph using our State structure
builder = StateGraph(State)

# Add the topic generator node to the graph
builder.add_node("topic_generator", topic_generator)

# Connect START to the topic generator
builder.add_edge(START, "topic_generator")

# Connect the topic generator to END
builder.add_edge("topic_generator", END)

graph=builder.compile()

In [10]:
result=graph.invoke({})

In [11]:
print(result)

{'topic': '**AI Topic:**  \n**Federated Learning for Privacy‑Preserving Medical Imaging Diagnostics**\n\n**Brief Description:**  \nInvestigate how federated learning (FL) can enable hospitals to collaboratively train deep‑learning models on sensitive medical imaging data (e.g., MRIs, CT scans) without sharing raw patient images. The research would focus on developing robust FL protocols that handle heterogeneous data distributions, communication constraints, and differential privacy guarantees, while maintaining diagnostic accuracy comparable to centralized training. The study could also explore model personalization for individual institutions and evaluate the system’s resilience to adversarial attacks.'}


### STATE MANAGEMENT

In [12]:
# Define the topic
topic = "Machine Learning"

# Generate an article about the topic
article = llm.invoke(f"Write about {topic}").content

# Summarize the generated article
summary = llm.invoke(f"Summarize this\n{article}").content

# Print the final summary
print(summary)

**Machine Learning: Quick Overview**

- **What it is** – ML builds models that learn patterns from data, enabling them to make decisions with little human rule‑writing.  
- **Historical milestones** – From the perceptron (1950s) → back‑propagation (1980s) → SVMs (1990s) → random forests (2000s) → deep CNNs (2010s) → transformers (2020s).  
- **Core ideas** –  
  - **Data** (labeled, unlabeled, semi‑labeled)  
  - **Model** (neural net, tree, kernel, etc.)  
  - **Training** (optimize loss with SGD/Adam)  
  - **Evaluation** (accuracy, precision, recall, ROC‑AUC)  
  - **Generalization** (performance on unseen data).  
- **Learning paradigms** –  
  - Supervised (classification, regression)  
  - Unsupervised (clustering, dimensionality reduction)  
  - Semi‑supervised & self‑supervised (pre‑training)  
  - Reinforcement (policy learning)  
  - Federated (privacy‑preserving).  
- **Key algorithms & architectures** – linear models, tree ensembles, SVMs, MLPs, CNNs, RNNs, Transformers, GN

In [13]:
# Define the structure of the graph state
class State(TypedDict):
    topic: str
    article: str
    summary: str

# Define the writer node
def writer(state):
    # Generate an article using the topic from the state
    article = llm.invoke(f"Write about {state['topic']}").content
    # Update the article in the state
    return {"article": article}

# Define the summarizer node
def summarizer(state):
    # Summarize the article from the state
    summary = llm.invoke(f"Summarize this\n{state['article']}").content
    # Update the summary in the state
    return {"summary": summary}

# Create the StateGraph
builder = StateGraph(State)

# Add nodes to the graph
builder.add_node("writer", writer)
builder.add_node("summarizer", summarizer)

# Define the flow of the graph
builder.add_edge(START, "writer")
builder.add_edge("writer", "summarizer")
builder.add_edge("summarizer", END)

# Compile the graph
graph = builder.compile()

In [14]:
# Run the graph with "Machine Learning" as the input topic
result = graph.invoke({"topic": "Machine Learning"})

# Print the final summary generated by the summarizer node
print(result["summary"])

**Machine Learning (ML) – Quick Overview**

- **What it is**: ML builds models that learn patterns from data, enabling automated decision‑making without hand‑coded rules.

---

### 1. Historical Milestones
| Era | Key Breakthrough | Impact |
|-----|------------------|--------|
| 1950s | Perceptron | First neural network |
| 1960s | Decision Trees | Rule‑based learning |
| 1980s | Backpropagation | Revived deep nets |
| 1990s | SVM | Powerful kernel methods |
| 2000s | Random Forests | Robust ensemble learning |
| 2010s | CNNs (AlexNet) | Image‑recognition boom |
| 2020s | Transformers (BERT, GPT) | NLP dominance, now multimodal |

---

### 2. Core Concepts
- **Data** – labeled, unlabeled, or semi‑labeled.  
- **Model** – mathematical mapping (e.g., NN, tree).  
- **Training** – adjust parameters to minimize a **loss function**.  
- **Optimization** – SGD, Adam, etc.  
- **Evaluation** – accuracy, precision, recall, ROC‑AUC.  
- **Generalization** – performance on unseen data.

---

###

### CONDITIONAL ROUTING

In [15]:
# Example question to test the routing logic
question = "Write Python code"

# Check whether the question contains the word "code"
if "code" in question.lower():
    print("Coding Agent")
else:
    print("General Agent")

# Define the structure of the graph state
class State(TypedDict):
    question: str
    route: str
    answer: str

# Define the router node
def router(state):
    # By default, route the question to the coding agent
    route = "coding"

    # If "code" is not present, route to the general agent
    if "code" not in state["question"].lower():
        route = "general"

    # Store the selected route in the state
    return {"route": route}

# Define the coding agent node
def coding_agent(state):
    # Return response from the coding agent
    return {"answer": "Coding Agent Selected"}

# Define the general agent node
def general_agent(state):
    # Return response from the general agent
    return {"answer": "General Agent Selected"}

Coding Agent


In [16]:
# Define the decision function
# It returns the route selected by the router node
def decide(state):
    return state["route"]

# Create the StateGraph
builder = StateGraph(State)

# Add all nodes to the graph
builder.add_node("router", router)
builder.add_node("coding", coding_agent)
builder.add_node("general", general_agent)

# Start the graph with the router node
builder.add_edge(START, "router")

# Route conditionally based on the value returned by decide()
builder.add_conditional_edges(
    "router",
    decide,
    {
        "coding": "coding",
        "general": "general"
    }
)

# End the graph after either agent finishes
builder.add_edge("coding", END)
builder.add_edge("general", END)

# Compile the graph
graph = builder.compile()

In [17]:
# Run the graph with a coding-related question
result = graph.invoke({"question": "Write Python code"})

# Print the answer returned by the selected agent
print(result["answer"])

Coding Agent Selected


### MESSAGE STATE

In [18]:
# First question tells the LLM your name
question1 = "My name is AB"

# Send the first question to the LLM
llm.invoke(question1)

# Ask the LLM to remember your name
question2 = "What is my name"

# Get the response from the LLM
answer = llm.invoke(question2).content

# Print the answer
print(answer)

I don’t have that information. Could you let me know what you’d like me to call you?


In [19]:
# Import MessagesState for storing conversation messages
from langgraph.graph import MessagesState

# Import HumanMessage for creating user messages
from langchain_core.messages import HumanMessage

In [20]:
# Define the chatbot node
def chatbot(state):
    # Send all messages stored in the state to the LLM
    response = llm.invoke(state["messages"])

    # Add the LLM response to the message history
    return {"messages": [response]}

In [21]:
# Create a StateGraph using MessagesState
builder = StateGraph(MessagesState)

# Add the chatbot node
builder.add_node("chatbot", chatbot)

# Define the graph flow
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# Compile the graph
graph = builder.compile()

In [22]:
# Send the first message to the graph
result = graph.invoke({
    "messages": [
        HumanMessage(content="My name is AB")
    ]
})

# Print the latest response from the chatbot
print(result["messages"][-1].content)

Nice to meet you, AB! How can I help you today?


In [23]:
# Send both messages together so the LLM has the conversation context
result = graph.invoke({
    "messages": [
        HumanMessage(content="My name is AB"),
        HumanMessage(content="What is my name")
    ]
})

# Print the latest response
print(result["messages"][-1].content)

Your name is AB.


In [24]:
# Send only this message in a new graph invocation
# The graph will NOT remember the previous invocation yet
result = graph.invoke({
    "messages": [
        HumanMessage(content="What is my name")
    ]
})

# Print the latest response
print(result["messages"][-1].content)

I don’t have that information. Could you let me know what you’d like me to call you?


### BUILDING A CHATBOT

In [25]:
# Take a question as input from the user
question = input("You: ")

# Send the question to the LLM
response = llm.invoke(question).content

# Print the LLM response
print(response)

The capital of India is **New Delhi**.


In [26]:
# Define the chatbot node
def chatbot(state):
    # Send the conversation messages to the LLM
    response = llm.invoke(state["messages"])

    # Add the LLM response to the message history
    return {"messages": [response]}

In [27]:
# Create a StateGraph using MessagesState
builder = StateGraph(MessagesState)

# Add the chatbot node
builder.add_node("chatbot", chatbot)

# Connect START to chatbot
builder.add_edge(START, "chatbot")

# Connect chatbot to END
builder.add_edge("chatbot", END)

# Compile the graph
graph = builder.compile()

In [28]:
# Import HumanMessage to create user messages
from langchain_core.messages import HumanMessage

In [30]:
# Create an empty list to store the complete conversation
conversation = []

# Start an infinite chat loop
while True:
    # Take input from the user
    user_input = input("You: ")

    # Stop the chatbot if the user types "exit"
    if user_input.lower() == "exit":
        break

    # Add the user's message to the conversation history
    conversation.append(HumanMessage(content=user_input))

    # Send the complete conversation history to the graph
    result = graph.invoke({"messages": conversation})

    # Update conversation with all messages including the AI response
    conversation = result["messages"]

    # Print the latest AI response
    print("AI:", conversation[-1].content)

AI: Nice to meet you, AB! How can I help you today?
AI: Your name is AB.


### Tool Calling with Nodes

In [31]:
# Ask the LLM about the current weather
# Note: LLM alone does not have access to live weather data
response = llm.invoke("What is the weather in Mumbai right now").content

# Print the response
print(response)

I don’t have real‑time data, so I can’t tell you the exact weather in Mumbai right now. For the most accurate and up‑to‑date information, check a reliable weather service such as:

- **Weather.com** (The Weather Channel)
- **AccuWeather**
- **Google Weather** (search “Mumbai weather”)
- **National Meteorological Department** (India)

These sites will give you current temperature, humidity, wind speed, chance of rain, and a short forecast.


In [32]:
# Import requests to call the free weather API
import requests

# Define the structure of the graph state
class State(TypedDict):
    city: str
    weather: str

In [33]:
# Define the weather node
def weather_node(state):
    # Free Open-Meteo geocoding API
    url = "https://geocoding-api.open-meteo.com/v1/search"

    # Find latitude and longitude of the city
    geo = requests.get(
        url,
        params={
            "name": state["city"],
            "count": 1
        }
    ).json()

    # Extract latitude and longitude
    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]

    # Get current weather using the free Open-Meteo API
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m"
        }
    ).json()

    # Extract current temperature
    temp = weather["current"]["temperature_2m"]

    # Store the temperature in the state
    return {"weather": f"{temp}°C"}

In [34]:
# Create the StateGraph
builder = StateGraph(State)

# Add the weather node to the graph
builder.add_node("weather", weather_node)

# Connect START to the weather node
builder.add_edge(START, "weather")

# Connect the weather node to END
builder.add_edge("weather", END)

# Compile the graph
graph = builder.compile()

In [35]:
# Run the graph with Mumbai as the city
result = graph.invoke({"city": "Mumbai"})

# Print the current temperature
print(result["weather"])

27.3°C


In [36]:
# Define the structure of the graph state
class State(TypedDict):
    city: str
    weather: str
    answer: str

In [37]:
# Define the weather node
def weather_node(state):
    # Free Open-Meteo geocoding API
    url = "https://geocoding-api.open-meteo.com/v1/search"

    # Find latitude and longitude of the city
    geo = requests.get(
        url,
        params={
            "name": state["city"],
            "count": 1
        }
    ).json()

    # Extract latitude and longitude
    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]

    # Get current weather using the free Open-Meteo API
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m"
        }
    ).json()

    # Extract current temperature
    temp = weather["current"]["temperature_2m"]

    # Store the temperature in the state
    return {"weather": f"{temp}°C"}

In [38]:
# Define the formatter node
def formatter(state):
    # Convert the raw weather data into a user-friendly response
    answer = llm.invoke(
        f"Convert this weather data into a user friendly response: {state['weather']}"
    ).content

    # Store the formatted response in the state
    return {"answer": answer}

In [39]:
# Create the StateGraph
builder = StateGraph(State)

# Add the weather node to get weather data
builder.add_node("weather", weather_node)

# Add the formatter node to create a user-friendly response
builder.add_node("formatter", formatter)

# Start the graph with the weather node
builder.add_edge(START, "weather")

# Send weather data to the formatter node
builder.add_edge("weather", "formatter")

# End the graph after formatting the response
builder.add_edge("formatter", END)

# Compile the graph
graph = builder.compile()